In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(color_codes=True)

legal_columns_to_display = [
    "Patient",
    "Nbre de lames",
    "Taille (cm)",
    "Valeur exacte AFP pré-opératoire",
    "Âge",
    "Genre masculin",
    "Nombre de nodules",
    "Expansif multinodulaire",
    "Nodule satellite",
    "Stéatohépatique",
    "Cellules claires",
    "MTM",
    "Squirrheux",
    "Riche en lymphocytes",
    "Riche en PNN",
    "Alcool",
    "NASH",
    "Hépatite B",
    "Hépatite C",
    "Autre étiologie",
    "Foie sain",
    "Albumine (N=35-50)",
    "Bilirubine T (N=<17)",
    "Grade ALBI",
    "BCLC stage",
    "Grade OMS",
    "mIV",
    "Naissance Afrique",
    "Naissance Amérique",
    "Naissance Asie",
    "Naissance Europe",
    "Récidive avant 2 ans",
    "Récidive dans la 1ère année",
    "Récidive entre 1 et 2 ans",
    "Récidive intra-hépatique",
    "Récidive extra-hépatique",
    "Récidive intra- et extra-hépatique",
]

In [ ]:
dfs = pd.read_excel("../data/tabs/table_prognosis.xlsx", sheet_name=None)

df_pb = dfs[list(dfs.keys())[0]][legal_columns_to_display]
df_hm = dfs[list(dfs.keys())[1]][legal_columns_to_display]
df_bj = dfs[list(dfs.keys())[2]][legal_columns_to_display]

In [ ]:
df_pb.head(7)

In [ ]:
df_hm.head(7)

In [ ]:
df_bj.head(7)

In [ ]:
cols_to_keep = [
    "Patient",
    "Nbre de lames",
    "Taille (cm)",
    "Valeur exacte AFP pré-opératoire",
    "Âge",
    "Genre masculin",
    "Nombre de nodules",
    "Expansif multinodulaire",
    "Nodule satellite",
    "Naissance Afrique",
    "Naissance Amérique",
    "Naissance Asie",
    "Naissance Europe",
    "Récidive avant 2 ans",
]

In [ ]:
df_pb = df_pb[cols_to_keep]
df_hm = df_hm[cols_to_keep]
df_bj = df_bj[cols_to_keep]

## Recurrence per patient

In [ ]:
final_count = pd.DataFrame()
counts = (
    df_pb.value_counts(["Récidive avant 2 ans"])
    .rename("Number of patients")
    .reset_index()
)
counts["Hospital"] = "Paul-Brousse"
final_count = pd.concat([final_count, counts], ignore_index=True)
counts = (
    df_hm.value_counts(["Récidive avant 2 ans"])
    .rename("Number of patients")
    .reset_index()
)
counts["Hospital"] = "Henri-Mondor"
final_count = pd.concat([final_count, counts], ignore_index=True)
counts = (
    df_bj.value_counts(["Récidive avant 2 ans"])
    .rename("Number of patients")
    .reset_index()
)
counts["Hospital"] = "Beaujon"
final_count = pd.concat([final_count, counts], ignore_index=True)
final_count["Récidive avant 2 ans"] = final_count["Récidive avant 2 ans"].replace(
    {0: "No recurrence", 1: "Recurrence"}
)
final_count

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.barplot(
    x="Hospital",
    y="Number of patients",
    hue="Récidive avant 2 ans",
    data=final_count,
    palette=reversed(sns.color_palette("husl", 2)),
)
for p in ax.patches:
    if p.get_height() != 0:
        ax.annotate(
            f"{p.get_height():.0f}",
            (p.get_x() + p.get_width() / 2.0, p.get_height()),
            ha="center",
            va="center",
            xytext=(0, 5),
            textcoords="offset points",
            fontsize=12,
        )
plt.legend(loc="upper right")
plt.ylabel("Number of patients", fontsize=12)
plt.xlabel("Hospital", fontsize=12)
plt.tight_layout()
plt.show()

## EDA

In [ ]:
fig, ax = plt.subplots(ncols=3, nrows=1, figsize=(18, 5), sharex=True)
sns.boxplot(x=df_pb["Taille (cm)"], ax=ax[0], color="crimson")
sns.boxplot(x=df_hm["Taille (cm)"], ax=ax[1], color="seagreen")
sns.boxplot(x=df_bj["Taille (cm)"], ax=ax[2], color="steelblue")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True, sharex=True)
# Paul-Brousse
counts_pb = (
    df_pb["Nombre de nodules"].astype(int).value_counts(normalize=True).sort_index()
)
counts_pb.plot(kind="bar", ax=axes[0], color="crimson")
axes[0].set_title("Paul-Brousse")
axes[0].set_xlabel("Number of nodules")
axes[0].set_ylabel("Proportion")
# Henri-Mondor
counts_hm = df_hm["Nombre de nodules"].value_counts(normalize=True).sort_index()
counts_hm.plot(kind="bar", ax=axes[1], color="seagreen")
axes[1].set_title("Henri-Mondor")
axes[1].set_xlabel("Number of nodules")
# Beaujon
counts_bj = df_bj["Nombre de nodules"].value_counts(normalize=True).sort_index()
counts_bj.plot(kind="bar", ax=axes[2], color="steelblue")
axes[2].set_title("Beaujon")
axes[2].set_xlabel("Number of nodules")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 7), sharex="row")
# ---------- Row 1: raw AFP ----------
axes[0, 0].hist(df_pb["Valeur exacte AFP pré-opératoire"], color="crimson")
axes[0, 0].set_title("Paul-Brousse")
axes[0, 1].hist(df_hm["Valeur exacte AFP pré-opératoire"], color="seagreen")
axes[0, 1].set_title("Henri-Mondor")
axes[0, 2].hist(df_bj["Valeur exacte AFP pré-opératoire"], color="steelblue")
axes[0, 2].set_title("Beaujon")

# ---------- Row 2: log1p(AFP) ----------
sns.histplot(
    np.log1p(df_pb["Valeur exacte AFP pré-opératoire"]),
    kde=True,
    ax=axes[1, 0],
    color="crimson",
)
sns.histplot(
    np.log1p(df_hm["Valeur exacte AFP pré-opératoire"]),
    kde=True,
    ax=axes[1, 1],
    color="seagreen",
)
sns.histplot(
    np.log1p(df_bj["Valeur exacte AFP pré-opératoire"]),
    kde=True,
    ax=axes[1, 2],
    color="steelblue",
)
for j in range(3):
    axes[0, j].set_xlabel("AFP")
    axes[1, j].set_xlabel("log1p(AFP)")
axes[0, 0].set_ylabel("count")
axes[1, 0].set_ylabel("density")
plt.tight_layout()
plt.show()